In [ ]:
import json
from pathlib import Path
from imgnet.collections.store import IndexedDatasets
from imgnet.collections.source import ZenodoSource
from imgtools.nifti.crawl import Crawler
from rich import print

In [ ]:
indexed_datasets = IndexedDatasets("indexed_datasets") # This will download the latest version of the indexed_datasets if you dont provide a path
print(indexed_datasets.collections[:10])

In [ ]:
description = """
Despite the considerable progress in automatic abdominal multi-organ segmentation from CT/MRI scans in recent years, a comprehensive evaluation of the models' capabilities is hampered by the lack of a large-scale benchmark from diverse clinical scenarios. Constraint by the high cost of collecting and labeling 3D medical data, most of the deep learning models to date are driven by datasets with a limited number of organs of interest or samples, which still limits the power of modern deep models and makes it difficult to provide a fully comprehensive and fair estimate of various methods. To mitigate the limitations, we present AMOS, a large-scale, diverse, clinical dataset for abdominal organ segmentation. <u>AMOS provides 500 CT and 100 MRI scans collected from multi-center, multi-vendor, multi-modality, multi-phase, multi-disease patients, each with voxel-level annotations of 15 abdominal organs, providing challenging examples and test-bed for studying robust segmentation algorithms</u> under diverse targets and scenarios. We further benchmark several state-of-the-art medical segmentation models to evaluate the status of the existing methods on this new challenging dataset. We have made our datasets, benchmark servers, and baselines publicly available, and hope to inspire future research.  For more details, please refer to our paper "https://arxiv.org/pdf/2206.08023.pdf" as well as homepage "https://jiyuanfeng.github.io/AMOS/".

AMOS provides the following content. imagesTr and labelsTr provide 240 scans (200 CT and 40 MRI), imagesVa and labelsVa provide 120 scans for model selection (100 CT and 20 MRI), and imagesTs provide 120 test data (please submit your predictions from https://amos22.grand-challenge.org/evaluation/challenge/submissions to get a score). Please note that id numbers less than 500 belong to CT data, otherwise they belong to MRI data.
"""

source = ZenodoSource(
    file_type="nifti",
    source="zenodo",
    record_id="7155725",
    description=description,
)


In [ ]:
dataset_index_path = (indexed_datasets.imgtools_path / "AMOS")
dataset_index_path.mkdir(parents=True, exist_ok=True)
with open(dataset_index_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)

In [ ]:
print(indexed_datasets.source_config("AMOS"))

In [ ]:
temp_data_path = Path("temp_data") / "AMOS"
temp_data_path.mkdir(parents=True, exist_ok=True)
downloader = indexed_datasets.downloader("AMOS").download(temp_data_path)

In [ ]:
import imgtools.loggers
import logging

imgtools.loggers.get_logger("imgtools").setLevel(logging.INFO)

crawler = Crawler(
    nifti_dir=temp_data_path,
    scan_name_pattern="amos22/images{Split}/amos_{ImageID:d}.nii.gz",
    mask_name_pattern="amos22/labels{Split}/amos_{ImageID:d}.nii.gz",
    output_dir=indexed_datasets.imgtools_path,
    n_jobs=10,
    force=True,
    deep=True,
)
crawler.crawl()

In [ ]:
index_df = indexed_datasets.index("AMOS")

In [ ]:
index_df

In [ ]:
def map_modality(row):
    if row["file_type"] == "scan":
        id = row["ImageID"]
        if id <= 500:
            return "CT"
        else:
            return "MR"
    elif row["file_type"] == "mask":
        return "SEG"

index_df["Modality"] = index_df.apply(map_modality, axis=1)
index_df

In [ ]:
index_df["BodyPartExamined"] = "ABDOMEN"

In [ ]:
import json

dataset_json = temp_data_path / "amos22" / "dataset.json"
with open(dataset_json, "r") as f:
    dataset = json.load(f)

dataset

In [ ]:
labels = dataset["labels"]
labels = ", ".join([v for k, v in labels.items() if v != "background"])
print(labels)


In [ ]:
def map_roinames(row):
    if row["file_type"] == "mask":
        return labels
    else:
        return ""

index_df["ROINames"] = index_df.apply(map_roinames, axis=1)
index_df


In [ ]:
index_df.to_csv(indexed_datasets.imgtools_path / "AMOS" / "index.csv", index=False)

In [ ]:
from imgnet.collections.store import IndexedDatasets

indexed_datasets = IndexedDatasets()

In [ ]:
from imgindex.model import validate_index

index = indexed_datasets.index("AMOS")
validate_index(index, "nifti", lazy=True)

In [ ]:
index.rename(columns={"reference_id": "SampleID"}, inplace=True)

In [ ]:
validate_index(index, "nifti", lazy=True)

In [ ]:
from pathlib import Path

new_index_dir = Path("/home/gpudual/bhklab/josh/med-image_index/new_index")
collection_path = new_index_dir / "AMOS"
collection_path.mkdir(parents=True, exist_ok=True)

index.to_csv(collection_path / "index.csv", index=False)

In [ ]:
import json
source = indexed_datasets.source_config("AMOS")

with open(collection_path / "source.json", "w") as f:
    json.dump(source.model_dump(mode="json"), f, indent=2)